In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')


In [2]:
#data Ingestion - scraping the data from a website
from langchain_community.document_loaders import WebBaseLoader

c:\python\OpenAI_AND_Ollama\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
#load data
loader=WebBaseLoader("https://www.geeksforgeeks.org/artificial-intelligence/introduction-to-langchain/")
docs=loader.load()

In [4]:
#divide loaded text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunked_documents=text_splitter.split_documents(docs)


In [5]:
chunked_documents

[Document(metadata={'source': 'https://www.geeksforgeeks.org/artificial-intelligence/introduction-to-langchain/', 'title': 'Introduction to LangChain - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Introduction to LangChain - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsIntroduction to LangChainLast Updated : 12 Dec, 2025LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for integrating with other tools and end-to-end chains for common applications. I

In [6]:
#loading the openai embeddings

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()


In [8]:
#converting the chuncks into vectors and storing in vectore store db

In [9]:
from langchain_community.vectorstores import FAISS
vectorestoredb=FAISS.from_documents(chunked_documents,embeddings)

In [10]:
#querying the vectordb
query="LangChain is an open-source framework designed to simplify the creation"
result=vectorestoredb.similarity_search(query)
result[0].page_content

'Introduction to LangChain - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsIntroduction to LangChainLast Updated : 12 Dec, 2025LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for integrating with other tools and end-to-end chains for common applications. It helps AI developers connect LLMs such as GPT-4 with external data and computation. This framework comes for both Python and JavaScript.Key benefits include: Modular Workflow: Simplifies chaining LLMs together for reusable and efficient workflows.Prompt Management: Offers tools for effective prompt engineering and memory handling.Ease of Integration: Streamlines the process of building LLM-powered applications.Key Components of LangChainLets see various components o

In [11]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [12]:
#retrieval chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Anser the following question based only on the provided context:
    <context>
    {context}
    </context>
    """
)

#document chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Anser the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_ch

In [14]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What is Langchain?",
    "context":[Document(page_content="LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for integrating with other tools and end-to-end chains for common applications. It helps AI developers connect LLMs such as GPT-4 with external data and computation. This framework comes for both Python and JavaScript.")]
})

'LangChain is designed to simplify the creation of applications using large language models (LLMs), providing a standard interface for integration and end-to-end chains for common applications.'

In [16]:
#retriever

retriever=vectorestoredb.as_retriever()

In [18]:
from langchain_classic.chains import create_retrieval_chain
retriever_chain=create_retrieval_chain(retriever,document_chain)

In [19]:
#get the response from the LLM
response=retriever_chain.invoke({"input":"What is Langchain?"})
response['answer']

"What is LangChain and what are some of its key components?\n\nLangChain is an open-source framework designed to simplify the creation of applications that use large language models (LLMs). It offers a standard interface for integrating with other tools and provides end-to-end chains for common applications, which help connect LLMs like GPT-4 with external data and computation. LangChain is available for both Python and JavaScript.\n\nSome key components of LangChain include:\n\n1. **Chains**: These define sequences of actions that can involve querying an LLM, manipulating data, or interacting with external tools. Chains can be simple, involving a single LLM invocation, or multi-step, combining multiple LLMs or actions.\n\n2. **Prompt Management**: This component facilitates managing and customizing prompts through tools like PromptTemplates, aiding in effective prompt engineering and memory handling.\n\n3. **Agents**: Autonomous systems that take actions based on input data, such as c